# Lesson 11 - Agent-to-Agent (A2A) Protocol


## Setap


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Wetin be A2A Protocol?

Di **Agent-to-Agent (A2A) protocol** na open standard wey make AI agents fit yarn, 
find each oda, and work togeder — even if dem different framework dem design di agents or 
if different service dey host dem.

Main tins wey get inside:

- **Discovery** – Agents dey publish *Agent Card* wey explain wetin dem fit do, e make am 
  easy for oda agents (or orchestrators) to find correct specialist for one work.
- **Message Passing** – Agents go send structured message through one common protocol, so 
  request from one agent fit be undastand and complete by anoda one no matter how e 
  dem implement am inside.
- **Task Lifecycle** – A2A dey talk about states like *submitted*, *working*, *completed*, and 
  *failed*, wey dey give orchestrator full clear eye on how one delegated work dey go.

For dis lesson, we go dey simulate A2A style collaboration by join three travel agents wey
specialize for different tins for one workflow, wey every agent go contribute im skill and pass 


## Di Way Dem Dey Take Create Special Travel Agents


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Multi-Agent Collaboration via Workflow

We connect de tree agents dem inside one by-one work wey resemble A2A message passing:

1. **CurrencyExchangeAgent** dey receive di user request and e go provide currency guide.
2. **ActivityPlannerAgent** dey receive di better context and e go add activity recommendations.
3. **TravelManagerAgent** go join both inputs dem together into one last travel brief.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Understanding A2A for Production

For production environment, di A2A protocol dey unlock strong cross-service scenarios:

| Capability | Description |
|---|---|
| **Cross-framework interop** | One agent wey dem build wit one framework fit commot plenty tasks go give another agent wey dem build wit any oda A2A-compliant framework, dis one dey allow true cross-organization interoperability. |
| **Service boundaries** | Agents fit dey inside separate microservices, cloud regions, or even different organisations but still dey work together well. |
| **Dynamic discovery** | One orchestrator fit check Agent Card registry anytime to find di best specialist for any sub-task. |
| **Streaming & push notifications** | A2A dey support Server-Sent Events (SSE) to give real-time progress updates and push notifications for long-running tasks. |

Di workflow we build before na small, in-process version of dis pattern. For real
deployment, every agent go show one HTTP endpoint, go publish one Agent Card, and go communicate
through A2A JSON-RPC protocol.


## Summary

For dis lesson you learn:

1. **Wetin be A2A protocol** — na open standard wey dey for agent-to-agent discovery, messaging,
   and task management.
2. **How to create specialized agents** — Currency Exchange agent, Activity Planner agent,
   and Travel Manager orchestrator.
3. **How to wire agents into workflow** — using `WorkflowBuilder` to model sequential
   message passing between agents.
4. **How A2A dey work for production** — e fit make cross-framework, cross-service collaboration
   with dynamic discovery and streaming updates.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
